# PhysAR-IDS -- NB5 (v6): Defence 1 -- adversarial training, tested against attacks it has not seen

**What Defence 1 is.** The standard IDS (Optuna-tuned XGBoost, NB2) is retrained on its clean training data **plus an equal number of adversarial rows** (true labels): PGD examples from a surrogate (NB3's attack) and examples produced by the PPO evasion agents (NB4 / NB4b). The result is the "Robust IDS". This notebook rebuilds it on the v6 pipeline, five seeds, and answers the reviewer's request: *confirm the attack samples used for testing are new and were not used in training; if reused, the drop to 0% means nothing.*

**How train and test attacks are kept apart.**
1. **Different rows.** Training adversarial examples are made from `X_train` rows only; every test attack is made from `X_test` rows, which lie *later in time* (temporal split). A check below counts exact row matches between the augmented training set and each test set.
2. **Different attacker, where it matters.** PGD training examples come from a *fresh surrogate* trained here (seed + 1000). Test PGD is made two ways: (a) by NB3's saved surrogate (a different network: an attacker the defence has never met) and (b) by the same training surrogate on test rows (same attacker, new rows).
3. **Budgets never seen in training.** PPO examples in training use the 0.1 and 0.3 sigma agents; the 0.5 sigma, 1.0 sigma and unconstrained agents are only used for testing.
4. **An attacker that adapts to the defence.** The PGD and PPO attackers above were built against the *standard* model and are fixed, so they only measure transfer to the defended model. The decisive test is the **white-box tree-greedy attack of NB3c / NB3d run against the defended model's own trees** (cyber features, 0.1 / 0.3 / 0.5 / 1.0 sigma, converged search). That is what "robust" must survive.

**Compared on identical rows and attackers:** standard vs defended, 5 seeds, paired tests. Reported: clean weighted/macro F1 and per-class recall (cost of the defence), PGD and PPO full-test F1, per-class evasion (attack rows predicted benign), benign recall (false alerts), and the white-box evasion per class. Attack rows only are perturbed for PPO and white-box (benign rows keep their model's clean predictions); PGD perturbs every row, as in NB3.

**Not done here (and why).** The PPO agents are not retrained against the defended model: that adaptive black-box attacker belongs to NB6 (Attack 3), which will also include the autoencoder gate.

**Runtime.** Roughly 30 min per seed on the GPU (fits, surrogate, PGD, PPO rollouts, white-box searches); results are saved per seed. Needs from earlier notebooks: `phase1_data_v6.pkl`, `xgb_tuned_best_params_v6.json`, `optuna_xgboost_results_v6.pkl`, the saved PPO agents of NB4 / NB4b, and (optionally) NB3's `surrogate_mlp_v6_seed*.pt`.

## 0. Setup

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings, copy
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
try:
    from sb3_contrib import RecurrentPPO
    from stable_baselines3.common.callbacks import CheckpointCallback
except ImportError as e:
    raise ImportError('Install in a terminal (not in a cell): pip install stable-baselines3 sb3-contrib gymnasium --no-deps') from e
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False

# ---- PPO / environment settings (unchanged from the reference design) ----
EPS_BOUND = 0.3; MAX_STEPS = 10
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EPS_TOTAL_BUDGET = 0.1
CLIP_BUDGETS = {'1.0': 1.0, 'unconstrained': None}     # budgets at which agents are RETRAINED with range clipping (stage B)
SAVED_AGENTS = {'0.1': ('ppo_attack2_budgeted_{v}_seed{s}', 0.1), '0.2': ('ppo_attack2_budget0p2_{v}_seed{s}', 0.2), '0.3': ('ppo_attack2_budget0p3_{v}_seed{s}', 0.3),
                '0.5': ('ppo_attack2_budget0p5_{v}_seed{s}', 0.5), '1.0': ('ppo_attack2_budget1p0_{v}_seed{s}', 1.0), 'unconstrained': ('ppo_attack2_unconstrained_{v}_seed{s}', None)}
EVAL_CHUNK = 8192


# ---- Defence 1 (adversarial training) settings ----
PGD_EPS = 0.1; PGD_ITERS = 10                       # as NB3 / NB3b (all 93 features, alpha = eps / 4)
TRAIN_PPO_TAGS = ['0.1', '0.3']                     # PPO agents (NB4 / NB4b, trained against the STANDARD model) whose examples go into the training set
EVAL_PPO_TAGS = ['0.1', '0.3', '0.5', '1.0', 'unconstrained']   # budgets 0.5 / 1.0 / unconstrained were never seen in training
N_PPO_TRAIN_PER_CLASS = 8000
AUG_RATIO = 1.0                                     # adversarial training rows per clean row (the paper's 1:1)
# ---- white-box (tree-greedy, NB3c / NB3d) attack against both models ----
K_CAND = 12; MAX_ITERS = 60; CHUNK = 20000; BLOCK = 300_000
WB_BUDGETS = {'0.1': 0.1, '0.3': 0.3, '0.5': 0.5, '1.0': 1.0}
N_WB_PER_CLASS = 2000                               # attack rows per class searched (same rows for every seed, model and budget)

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME}')
import sb3_contrib, stable_baselines3
print(f'sb3-contrib {sb3_contrib.__version__} | stable-baselines3 {stable_baselines3.__version__} | gymnasium {gym.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data and the NB2 tuned configuration

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | attacker-controlled (cyber) {N_CYBER} | training pool {X_atk_train.shape} | evaluation rows {X_atk_test.shape} in {len(np.unique(test_burst))} attack bursts')
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. Oracle and the batched PPO evaluator (NB4 code)

In [ ]:
class Oracle:
    """Hard-label black-box oracle over the fitted XGBoost classifier."""
    def __init__(self, clf): self.clf = clf; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X)
        return self.clf.predict(X)

def apply_action(x_cur, x_orig, action, eps_bound=EPS_BOUND, eps_total=None, range_clip=False):
    """In place on x_cur[:, CYBER_IDX]; returns the applied (post-clamp) perturbation."""
    a = np.clip(action, -eps_bound, eps_bound).astype(np.float32)
    old = x_cur[:, CYBER_IDX].copy(); prop = old + a
    if eps_total is not None:
        o = x_orig[:, CYBER_IDX]; prop = o + np.clip(prop - o, -eps_total, eps_total)
    if range_clip:   # never leave the observed training range [min, max] (a value already outside it may stay where it was, but may not go further out)
        prop = np.clip(prop, np.minimum(FEAT_LO[CYBER_IDX], x_orig[:, CYBER_IDX]), np.maximum(FEAT_HI[CYBER_IDX], x_orig[:, CYBER_IDX]))
    x_cur[:, CYBER_IDX] = prop
    return (prop - old).astype(np.float32)

def build_obs(x_cur, last_applied, last_pred, step):
    oh = np.zeros((len(x_cur), 6), dtype=np.float32); oh[np.arange(len(x_cur)), last_pred] = 1.0
    return np.concatenate([x_cur.astype(np.float32), last_applied.astype(np.float32), oh, np.full((len(x_cur), 1), step / MAX_STEPS, dtype=np.float32)], 1)

class IDSEvasionEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, X_pool, y_pool, oracle, eps_bound=EPS_BOUND, max_steps=MAX_STEPS, seed=None, eps_total_budget=None, range_clip=False):
        super().__init__(); self.range_clip = range_clip
        self.X_pool, self.y_pool, self.oracle = X_pool, y_pool, oracle
        self.eps_bound, self.max_steps, self.eps_total_budget = eps_bound, max_steps, eps_total_budget
        self.rng = np.random.RandomState(seed)
        self.action_space = spaces.Box(low=-eps_bound, high=eps_bound, shape=(N_CYBER,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(N_FEAT + N_CYBER + 6 + 1,), dtype=np.float32)
    def _obs(self): return build_obs(self.x_current[None], self.last_applied[None], np.array([self.last_pred]), self.step_count)[0]
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.rng.randint(0, len(self.X_pool))
        self.x_orig = self.X_pool[idx].copy(); self.x_current = self.x_orig.copy(); self.y_true = self.y_pool[idx]
        self.step_count = 0; self.last_applied = np.zeros(N_CYBER, dtype=np.float32)
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); return self._obs(), {}
    def step(self, action):
        applied = apply_action(self.x_current[None], self.x_orig[None], np.asarray(action, dtype=np.float32)[None], self.eps_bound, self.eps_total_budget, self.range_clip)[0]
        self.last_applied = applied; self.step_count += 1
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); pen = STEP_PENALTY_SCALE * float(np.sum(applied ** 2))
        terminated = truncated = False
        if self.last_pred == 0: reward = EVASION_REWARD - pen; terminated = True
        elif self.step_count >= self.max_steps: reward = TIMEOUT_PENALTY - pen; truncated = True
        else: reward = -pen
        return self._obs(), reward, terminated, truncated, {'evaded': self.last_pred == 0}

def run_batched(policy, X0, oracle, eps_total, mode='ppo', rng=None, range_clip=False):
    """Run one episode per row of X0 in parallel (10 steps max). mode: 'ppo' (deterministic policy) or 'random'."""
    N = len(X0); x_cur = X0.copy(); last_applied = np.zeros((N, N_CYBER), dtype=np.float32)
    last_pred = np.concatenate([oracle.predict(X0[i:i + 65536]) for i in range(0, N, 65536)]).astype(np.int64); clean_pred = last_pred.copy()
    active = np.ones(N, bool); evaded = np.zeros(N, bool); steps = np.zeros(N, np.int64)
    if mode == 'ppo': h = np.zeros((1, N, 64), np.float32); c = np.zeros((1, N, 64), np.float32)
    for step in range(MAX_STEPS):
        idx = np.where(active)[0]
        if len(idx) == 0: break
        obs = build_obs(x_cur[idx], last_applied[idx], last_pred[idx], step)
        if mode == 'ppo':
            act = np.zeros((len(idx), N_CYBER), np.float32)
            for s in range(0, len(idx), EVAL_CHUNK):
                sl = idx[s:s + EVAL_CHUNK]
                a, (hn, cn) = policy.predict(obs[s:s + EVAL_CHUNK], state=(h[:, sl], c[:, sl]), episode_start=np.full(len(sl), step == 0), deterministic=True)
                act[s:s + EVAL_CHUNK] = a; h[:, sl] = hn; c[:, sl] = cn
        else:
            act = rng.uniform(-EPS_BOUND, EPS_BOUND, size=(len(idx), N_CYBER)).astype(np.float32)
        sub = x_cur[idx]; applied = apply_action(sub, X0[idx], act, EPS_BOUND, eps_total, range_clip); x_cur[idx] = sub; last_applied[idx] = applied
        pred = np.concatenate([oracle.predict(sub[i:i + 65536]) for i in range(0, len(sub), 65536)]).astype(np.int64); last_pred[idx] = pred
        ev = pred == 0; evaded[idx[ev]] = True; steps[idx[ev]] = step + 1; active[idx[ev]] = False
    return {'evaded': evaded, 'steps': steps, 'x_final': x_cur, 'pred_final': last_pred, 'clean_pred': clean_pred}

def per_class(vec_bool):
    return np.array([vec_bool[y_atk_test == c].mean() for c in range(1, 6)])
def mean_steps(res):
    return np.array([res['steps'][(y_atk_test == c) & res['evaded']].mean() if ((y_atk_test == c) & res['evaded']).any() else np.nan for c in range(1, 6)])
def summarise(res):
    disp = np.abs(res['x_final'][:, CYBER_IDX] - X_atk_test[:, CYBER_IDX])
    _ex = lambda Z: np.maximum(np.maximum(FEAT_LO[CYBER_IDX] - Z[:, CYBER_IDX], Z[:, CYBER_IDX] - FEAT_HI[CYBER_IDX]), 0).max(1)
    exc = np.maximum(_ex(res['x_final']) - _ex(X_atk_test), 0)    # sigma ADDED beyond what the clean test row already had outside the training min/max
    return {'per_class_evasion': per_class(res['evaded']), 'overall_evasion': float(res['evaded'].mean()), 'mean_steps_to_evade': mean_steps(res),
            'mean_queries': float(1 + np.where(res['evaded'], res['steps'], MAX_STEPS).mean()),
            'attack_f1': float(f1_score(y_atk_test, res['pred_final'], average='weighted')),
            'max_abs_disp_mean': float(disp.max(1).mean()), 'max_abs_disp_max': float(disp.max()), 'mean_excess_beyond_train_range': float(exc.mean()), 'frac_rows_beyond_range_0.5sigma': float((exc > 0.5).mean()),
            'evaded': res['evaded'].copy()}
print('environment + evaluator defined; cyber features controlled:', N_CYBER)

## 3. Full-test metrics (NB4e)

In [ ]:
from sklearn.metrics import precision_score, recall_score
LAB = list(range(6))
def full_metrics(pred_atk, benign_pred):
    """Full-test metrics when the attack rows carry predictions `pred_atk` and benign rows keep `benign_pred`."""
    pred = np.empty(len(y_test), dtype=np.int64); pred[y_test == 0] = benign_pred; pred[atk_te] = pred_atk
    return {'weighted_f1': float(f1_score(y_test, pred, average='weighted')), 'macro_f1': float(f1_score(y_test, pred, average='macro')),
            'per_class_f1': f1_score(y_test, pred, average=None, labels=LAB), 'per_class_recall': recall_score(y_test, pred, average=None, labels=LAB),
            'benign_precision': float(precision_score(y_test, pred, labels=[0], average=None)[0]), 'attack_detected': float((pred_atk != 0).mean())}
print('full_metrics defined; full test rows', len(y_test), '| attack rows', int(atk_te.sum()))

## 4. White-box tree-greedy attack (NB3c / NB3d code)

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access). Scores in blocks and, if the GPU runs out of memory, halves the block and retries."""
    def __init__(self, clf): self.b = clf.get_booster(); self.bs = BLOCK
    def _pred(self, X):
        while True:
            try:
                return [np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + self.bs], dtype=np.float32), predict_type='margin', validate_features=False)) for s in range(0, len(X), self.bs)]
            except xgb.core.XGBoostError as e:
                if 'out of memory' not in str(e).lower() or self.bs <= 20000: raise
                self.bs = max(20000, self.bs // 2); print(f'      GPU out of memory -> scoring in blocks of {self.bs:,} rows', flush=True)
    def __call__(self, X):
        out = self._pred(X); return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox_trace(margin, X0, y0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    """as NB3c's whitebox, plus a trace: per round, the evaded share per class (classes 4, 5) and the median margin gap of the rows not yet evaded."""
    x = X0.copy(); g = gap_of(margin(x)); active = g <= 0; stalled = np.zeros(len(x), bool); trace = []; t_start = time.time()
    active = ~(g > 0); init_ev = g > 0
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); ev = g > 0
        trace.append({'round': it + 1, 'active': int(len(act)), 'evaded_c4': float(ev[y0 == 4].mean()), 'evaded_c5': float(ev[y0 == 5].mean()),
                      'median_gap_unevaded': float(np.median(g[~ev])) if (~ev).any() else float('nan'), 'seconds': time.time() - t0})
        if verbose and ((it + 1) % 10 == 0 or it == 0): print(f'      round {it + 1:>3}: active {len(act):>5} | evaded ransomware {trace[-1]["evaded_c4"]:.3f} reverseshell {trace[-1]["evaded_c5"]:.3f} | median gap of unevaded {trace[-1]["median_gap_unevaded"]:+.3f} | {time.time() - t0:4.1f}s', flush=True)
    return x, g, init_ev, stalled, trace
def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):      # used by the self-test
    return whitebox_trace(margin, X0, np.zeros(len(X0), dtype=np.int64), feats, TH, B, K, max_iters, chunk, verbose)
print('white-box attack defined')

In [ ]:
def selftest(clf, margin, TH, seed=0):
    rng = np.random.RandomState(seed); rows = rng.choice(len(X_atk_test), 60, replace=False); X0 = X_atk_test[rows]; feats = rng.choice(CYBER_IDX, 12, replace=False)
    g0 = gap_of(margin(X0)); worst = 0.0
    for B in [0.1, 0.5, None]:
        bg, bj, bv, _ = one_step(margin, X0.copy(), X0, g0, feats, TH, B, K=10 ** 6)
        for i in range(len(rows)):
            best = g0[i]
            for j in feats:
                T = TH.get(int(j))
                if T is None: continue
                a, b = interval(X0[i:i + 1, j], int(j), B); a, b = float(a[0]), float(b[0]); inner = T[(T > a) & (T < b)]
                pts = np.concatenate([[a, b], (inner[:-1] + inner[1:]) / 2 if len(inner) > 1 else [], inner, np.nextafter(inner, np.float32(-np.inf))]).astype(np.float32)
                pts = pts[(pts >= a) & (pts <= b)]
                Xc = np.repeat(X0[i:i + 1], len(pts), 0); Xc[:, j] = pts; best = max(best, gap_of(margin(Xc)).max())
            worst = max(worst, best - bg[i])
            assert bg[i] >= best - 1e-4, f'one_step missed a better move (row {i}, budget {B}): {bg[i]} < {best}'
    # constraints after a short attack
    sub = X_atk_test[rng.choice(len(X_atk_test), 400, replace=False)]
    for B in [0.1, 0.5]:
        xa, ga, ie, st, lg = whitebox(margin, sub, CYBER_IDX, TH, B, max_iters=4, verbose=False)
        d = np.abs(xa - sub); assert d[:, CYBER_IDX].max() <= B + 1e-6, 'L-inf budget violated'
        other = np.setdiff1d(np.arange(N_FEAT), CYBER_IDX); assert d[:, other].max() == 0, 'a non-controllable feature changed'
        lo_ok = (xa >= np.minimum(FEAT_LO, sub) - 1e-6) & (xa <= np.maximum(FEAT_HI, sub) + 1e-6); assert lo_ok.all(), 'left the training range'
        assert (ga >= gap_of(margin(sub)) - 1e-9).all(), 'margin gap decreased'
    print(f'self-test passed (largest shortfall of one_step vs full enumeration: {max(worst, 0):.2e}); budget, range, non-controllable features and monotone margin all hold')

## 5. Surrogate, PGD and the training-set builder
`train_surrogate` and `pgd` follow NB3 / NB3b exactly (93-64-32-6 MLP, 20 epochs, Adam 1e-3, true labels; PGD on all 93 features, eps 0.1, alpha = eps/4, 10 steps). PGD is run in blocks to keep GPU memory low.

In [ ]:
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)
def train_surrogate(seed, X_tr, y_tr, n_epochs=20):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed); torch.cuda.manual_seed_all(seed)
    model = SurrogateMLP(X_tr.shape[1]).to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    ds = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).long()); loader = DataLoader(ds, batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(seed))
    for ep in range(n_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
    model.eval(); return model
def pgd(model, X, y, eps=PGD_EPS, n_iter=PGD_ITERS, bs=65536):
    model.eval(); alpha = eps / 4; out = np.empty_like(X)
    for s in range(0, len(X), bs):
        yt = torch.from_numpy(y[s:s + bs]).long().to(DEVICE); X0 = torch.from_numpy(X[s:s + bs]).float().to(DEVICE); Xa = X0.clone()
        for _ in range(n_iter):
            Xa.requires_grad_(True); g = torch.autograd.grad(F.cross_entropy(model(Xa), yt), Xa)[0]; Xa = Xa.detach() + alpha * g.sign(); Xa = X0 + torch.clamp(Xa - X0, -eps, eps)
        out[s:s + bs] = Xa.detach().cpu().numpy()
    return out

def fit_pair(X, y, seed):
    """GPU-fitted classifier (used for the white-box margin search) and a CPU copy (the oracle for every reported number, as in NB4)."""
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X, y, verbose=False)
    tmp = f'_tmp_xgb_{seed}_{os.getpid()}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp); return clf, cpu

def ppo_train_examples(seed, oracle):
    rng = np.random.RandomState(seed + 77); Xs, ys, used = [], [], []
    for tag in TRAIN_PPO_TAGS:
        pat, b = SAVED_AGENTS[tag]; path = pat.format(v=PIPELINE_VERSION, s=seed) + '.zip'
        if not os.path.exists(path): print(f'    {path} not found -- skipped'); continue
        idx = np.concatenate([rng.choice(np.flatnonzero(y_atk_train == c), min(N_PPO_TRAIN_PER_CLASS, int((y_atk_train == c).sum())), replace=False) for c in range(1, 6)])
        res = run_batched(RecurrentPPO.load(path), X_atk_train[idx], oracle, b, mode='ppo'); Xs.append(res['x_final']); ys.append(y_atk_train[idx]); used.append((tag, float(res['evaded'].mean())))
    return (np.vstack(Xs), np.concatenate(ys), used) if Xs else (np.zeros((0, N_FEAT), np.float32), np.zeros(0, np.int64), used)

def row_keys(X): return {r.tobytes() for r in np.ascontiguousarray(X, dtype=np.float32)}
def pgd_eval(cpu_m, X_adv):
    pred = cpu_m.predict(X_adv); ev = pred[atk_te] == 0
    return {'per_class_evasion': per_class(ev), 'overall_evasion': float(ev.mean()), 'weighted_f1': float(f1_score(y_test, pred, average='weighted')), 'benign_recall': float((pred[y_test == 0] == 0).mean()), 'attack_detected': float((~ev).mean())}
print('defence components defined')

## 6. Rows for the white-box search and the self-test input

In [ ]:
rng0 = np.random.RandomState(0); pick = np.sort(np.concatenate([rng0.choice(np.flatnonzero(y_atk_test == c), N_WB_PER_CLASS, replace=False) for c in range(1, 6)]))
X_wb, y_wb = X_atk_test[pick], y_atk_test[pick]
def per_class_wb(ev): return np.array([ev[y_wb == c].mean() for c in range(1, 6)])
print(f'white-box rows: {len(X_wb)} ({N_WB_PER_CLASS} per attack class) | budgets {WB_BUDGETS} | up to {MAX_ITERS} rounds')

## 7. Per-seed run (standard model, adversarial training set, defended model, all attacks)

In [ ]:
R = {}; INFO = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'defence1_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = repr((best_params, PGD_EPS, TRAIN_PPO_TAGS, EVAL_PPO_TAGS, N_PPO_TRAIN_PER_CLASS, AUG_RATIO, K_CAND, MAX_ITERS, WB_BUDGETS, N_WB_PER_CLASS, seed, X_atk_test.shape, float(X_atk_test[:5].sum()), 'nb5-v1'))
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed], INFO[seed] = saved['R'], saved['info']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True); R[seed] = {}; info = {}
    # ---- standard model ----
    std_gpu, std_cpu = fit_pair(X_train, y_train, seed); oracle_std = Oracle(std_cpu); clean_std = std_cpu.predict(X_test)
    ref = nb2['results']['tuned'].get(seed, {}).get('test_f1', np.nan); info['clean_f1_standard'] = float(f1_score(y_test, clean_std, average='weighted'))
    print(f'  standard model clean F1 {info["clean_f1_standard"]:.4f} (NB2 recorded {ref:.4f})', flush=True)
    if seed == SEEDS[0]: selftest(std_gpu, Margin(std_gpu), get_thresholds(std_gpu))
    # ---- adversarial training set ----
    t1 = time.time(); surA = train_surrogate(seed + 1000, X_train, y_train); X_pgd_tr = pgd(surA, X_train, y_train)
    X_ppo_tr, y_ppo_tr, used = ppo_train_examples(seed, oracle_std); info['ppo_train_evasion_vs_standard'] = used
    X_pool = np.vstack([X_pgd_tr, X_ppo_tr]); y_pool = np.concatenate([y_train, y_ppo_tr]); n_adv = int(round(AUG_RATIO * len(X_train)))
    sel = np.random.RandomState(seed).choice(len(X_pool), size=min(n_adv, len(X_pool)), replace=False); X_aug = np.vstack([X_train, X_pool[sel]]); y_aug = np.concatenate([y_train, y_pool[sel]])
    perm = np.random.RandomState(seed).permutation(len(X_aug)); X_aug, y_aug = X_aug[perm], y_aug[perm]
    info['n_clean'] = len(X_train); info['n_adv'] = int(len(sel)); info['n_pgd_in_aug'] = int((sel < len(X_pgd_tr)).sum()); info['n_ppo_in_aug'] = int((sel >= len(X_pgd_tr)).sum())
    print(f'  training set: {len(X_train):,} clean + {len(sel):,} adversarial ({info["n_pgd_in_aug"]:,} PGD, {info["n_ppo_in_aug"]:,} PPO; PPO train evasion vs standard {used}) | built in {(time.time() - t1) / 60:.1f} min', flush=True)
    # ---- defended model ----
    def_gpu, def_cpu = fit_pair(X_aug, y_aug, seed); oracle_def = Oracle(def_cpu); clean_def = def_cpu.predict(X_test); info['clean_f1_defended'] = float(f1_score(y_test, clean_def, average='weighted'))
    print(f'  defended model clean F1 {info["clean_f1_defended"]:.4f} (cost {info["clean_f1_standard"] - info["clean_f1_defended"]:+.4f})', flush=True)
    def_gpu.save_model(f'xgb_defended_{PIPELINE_VERSION}_seed{seed}.json')
    models = {'std': (std_cpu, std_gpu, oracle_std, clean_std), 'def': (def_cpu, def_gpu, oracle_def, clean_def)}
    for k, (cm_, _, _, cp_) in models.items():
        m = full_metrics(cp_[atk_te], cp_[y_test == 0]); R[seed][f'clean|{k}'] = {**m, 'per_class_evasion': per_class(cp_[atk_te] == 0), 'overall_evasion': float((cp_[atk_te] == 0).mean())}
    # ---- no test row was used in training (exact-match check) ----
    keys_aug = row_keys(X_aug); test_keys = row_keys(X_test)
    info['exact_match_clean_test_rows_in_training_set'] = int(sum(r.tobytes() in keys_aug for r in np.ascontiguousarray(X_test, dtype=np.float32)))
    # ---- PGD test sets ----
    adv_sets = {}
    sp = f'surrogate_mlp_{PIPELINE_VERSION}_seed{seed}.pt'
    if os.path.exists(sp):
        sur = SurrogateMLP(N_FEAT).to(DEVICE); sur.load_state_dict(torch.load(sp, map_location=DEVICE)); sur.eval(); adv_sets['pgd_nb3'] = pgd(sur, X_test, y_test)
        if seed == SEEDS[0] and os.path.exists(f'X_adv_pgd_eps01_{PIPELINE_VERSION}.npy'):
            sv = np.load(f'X_adv_pgd_eps01_{PIPELINE_VERSION}.npy')
            if sv.shape == adv_sets['pgd_nb3'].shape: info['pgd_nb3_regen_vs_saved_max_abs_diff'] = float(np.abs(sv - adv_sets['pgd_nb3']).max()); info['pgd_nb3_regen_vs_saved_share_equal'] = float((sv == adv_sets['pgd_nb3']).mean())
    else: print(f'  {sp} not found -- NB3-surrogate PGD test skipped')
    adv_sets['pgd_seen'] = pgd(surA, X_test, y_test)
    info['exact_match_adv_test_rows_in_training_set'] = {k: int(sum(r.tobytes() in keys_aug for r in np.ascontiguousarray(v, dtype=np.float32))) for k, v in adv_sets.items()}
    for k, v in adv_sets.items():
        for mk, (cm_, _, _, _) in models.items(): R[seed][f'{k}|{mk}'] = pgd_eval(cm_, v)
    r_ = R[seed]; print('  PGD test (evasion on attack rows, std -> def): ' + ' | '.join(f'{k} {r_[f"{k}|std"]["overall_evasion"]:.3f} -> {r_[f"{k}|def"]["overall_evasion"]:.3f}' for k in adv_sets), flush=True)
    # ---- fixed PPO agents (trained against the standard model) re-run against each model ----
    for tag in EVAL_PPO_TAGS:
        pat, b = SAVED_AGENTS[tag]; path = pat.format(v=PIPELINE_VERSION, s=seed) + '.zip'
        if not os.path.exists(path): print(f'  {path} not found -- skipped'); continue
        agent = RecurrentPPO.load(path)
        for mk, (cm_, _, orc, cp_) in models.items():
            res = run_batched(agent, X_atk_test, orc, b, mode='ppo'); m = full_metrics(res['pred_final'], cp_[y_test == 0]); R[seed][f'ppo_{tag}|{mk}'] = {**m, 'per_class_evasion': per_class(res['pred_final'] == 0), 'overall_evasion': float((res['pred_final'] == 0).mean())}
        print(f'  PPO {tag:>13}: evasion std {R[seed][f"ppo_{tag}|std"]["overall_evasion"]:.3f} -> def {R[seed][f"ppo_{tag}|def"]["overall_evasion"]:.3f} | full-test F1 {R[seed][f"ppo_{tag}|std"]["weighted_f1"]:.4f} -> {R[seed][f"ppo_{tag}|def"]["weighted_f1"]:.4f}', flush=True)
    # ---- white-box attack on each model's own trees (adaptive) ----
    for mk, (cm_, gm, _, _) in models.items():
        mg = Margin(gm); TH = get_thresholds(gm)
        for tag, B in WB_BUDGETS.items():
            ts = time.time(); xa, ga, ie, st, tr = whitebox(mg, X_wb, CYBER_IDX, TH, B, verbose=False); ev = cm_.predict(xa) == 0
            R[seed][f'wb_{tag}|{mk}'] = {'per_class_evasion': per_class_wb(ev), 'overall_evasion': float(ev.mean()), 'rows_stalled': float((st & ~ev).mean()), 'rows_round_limit': float((~st & ~ev).mean()), 'search_vs_cpu_disagree': float(((ga > 0) != ev).mean()), 'rounds': len(tr)}
            print(f'  white-box {mk} {tag:>4}: per-class {np.round(R[seed][f"wb_{tag}|{mk}"]["per_class_evasion"], 3).tolist()} overall {ev.mean():.3f} | stalled {R[seed][f"wb_{tag}|{mk}"]["rows_stalled"]:.2f} round-limit {R[seed][f"wb_{tag}|{mk}"]["rows_round_limit"]:.3f} | {(time.time() - ts) / 60:.1f} min', flush=True)
    INFO[seed] = info; joblib.dump({'fp': cfg_fp, 'R': R[seed], 'info': info}, pf); print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 8. Is any test row also in the training set? (reviewer check)

In [ ]:
print('Exact feature-vector matches between each TEST set and the augmented TRAINING set (per seed):')
for s in SEEDS:
    i = INFO[s]; print(f'  seed {s}: clean test rows {i["exact_match_clean_test_rows_in_training_set"]} of {len(X_test):,} | adversarial test sets {i["exact_match_adv_test_rows_in_training_set"]}')
print('(A clean test row can only match a training row if the dataset itself contains identical measurements; adversarial test rows should match none.)')
if 'pgd_nb3_regen_vs_saved_max_abs_diff' in INFO[SEEDS[0]]: print(f'NB3 surrogate PGD regenerated here vs the saved X_adv_pgd_eps01 (seed {SEEDS[0]}): max |diff| {INFO[SEEDS[0]]["pgd_nb3_regen_vs_saved_max_abs_diff"]:.2e}, share of identical entries {INFO[SEEDS[0]]["pgd_nb3_regen_vs_saved_share_equal"]:.4f}')
print('PPO training-example evasion against the standard model:', {s: INFO[s]['ppo_train_evasion_vs_standard'] for s in SEEDS})

## 9. Results

In [ ]:
atk_names = class_names[1:]
def hv(k): return [s for s in SEEDS if k in R[s]]
def A(k, f): return np.stack([np.asarray(R[s][k][f]) for s in hv(k)])
def ptest(a, b):
    d = a - b
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (0.0 if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
ms = lambda x: f'{x.mean():.4f} +/- {(x.std(ddof=1) if len(x) > 1 else 0):.4f}'
# 9a. clean cost
cs, cd = A('clean|std', 'weighted_f1'), A('clean|def', 'weighted_f1'); ms_, md_ = A('clean|std', 'macro_f1'), A('clean|def', 'macro_f1'); rs_, rd_ = A('clean|std', 'per_class_recall'), A('clean|def', 'per_class_recall')
print('9a. Cost of the defence on CLEAN test data'); print(f'   weighted F1  standard {ms(cs)}  defended {ms(cd)}  change {(cd - cs).mean():+.4f} (p {ptest(cd, cs):.4f})'); print(f'   macro F1     standard {ms(ms_)}  defended {ms(md_)}  change {(md_ - ms_).mean():+.4f} (p {ptest(md_, ms_):.4f})')
print('   per-class recall (standard -> defended):', {c: f'{rs_[:, i].mean():.3f}->{rd_[:, i].mean():.3f}' for i, c in enumerate(class_names)})
# 9b. every attack: standard vs defended
ATTACKS = [('pgd_nb3', 'PGD, NB3 surrogate (unseen attacker)'), ('pgd_seen', 'PGD, training surrogate (seen attacker)')] + [(f'ppo_{t}', f'PPO {t} sigma' + (' (seen budget)' if t in TRAIN_PPO_TAGS else ' (unseen budget)')) for t in EVAL_PPO_TAGS] + [(f'wb_{t}', f'white-box tree-greedy {t} sigma (adaptive)') for t in WB_BUDGETS]
rows = []
for k, label in ATTACKS:
    if not (hv(f'{k}|std') and hv(f'{k}|def')): continue
    es, ed = A(f'{k}|std', 'overall_evasion'), A(f'{k}|def', 'overall_evasion'); row = {'attack': label, 'evasion standard': ms(es), 'evasion defended': ms(ed), 'change': f'{(ed - es).mean():+.4f}', 'p (paired)': f'{ptest(ed, es):.4f}'}
    if 'weighted_f1' in R[SEEDS[0]][f'{k}|std']:
        fs, fd = A(f'{k}|std', 'weighted_f1'), A(f'{k}|def', 'weighted_f1'); row['F1 standard'] = ms(fs); row['F1 defended'] = ms(fd); row['F1 change'] = f'{(fd - fs).mean():+.4f}'; row['F1 p (paired)'] = f'{ptest(fd, fs):.4f}'
    rows.append(row)
overall_table = pd.DataFrame(rows); pd.set_option('display.width', 320); print('\n9b. Overall evasion of attack rows, standard vs defended (mean +/- std over seeds)'); print(overall_table.to_string(index=False))
# 9c. per class
pc_rows = []
for k, label in ATTACKS:
    if not (hv(f'{k}|std') and hv(f'{k}|def')): continue
    a_s, a_d = A(f'{k}|std', 'per_class_evasion'), A(f'{k}|def', 'per_class_evasion'); raw = np.array([ptest(a_d[:, i], a_s[:, i]) for i in range(5)]); adj = holm(np.nan_to_num(raw, nan=1.0))
    pc_rows.append({'attack': label, **{c: f'{a_s[:, i].mean():.3f} -> {a_d[:, i].mean():.3f}' + ('*' if adj[i] < 0.05 else '') for i, c in enumerate(atk_names)}})
pc_table = pd.DataFrame(pc_rows); print('\n9c. Per-class evasion, standard -> defended (* = Holm-adjusted p < 0.05 over the 5 classes)'); print(pc_table.to_string(index=False))
# 9d. false alerts under PGD (PGD perturbs benign rows too)
fa = [{'attack': l, 'benign recall std': f'{A(f"{k}|std", "benign_recall").mean():.3f}', 'benign recall def': f'{A(f"{k}|def", "benign_recall").mean():.3f}'} for k, l in ATTACKS[:2] if hv(f'{k}|std')]
print('\n9d. Benign rows kept benign under PGD'); print(pd.DataFrame(fa).to_string(index=False) if fa else 'n/a')
# 9e. verdict per attack
print('\n9e. Verdict per attack (overall evasion, defended vs standard; paired over seeds)')
print('   (white-box rows have no full-test F1: that attack runs on a fixed subsample of attack rows only)')
for r in rows:
    ch, p = float(r['change']), float(r['p (paired)']) if r['p (paired)'] != 'nan' else 1.0
    print(f'   {r["attack"]:<48} {"HELPS" if ch < 0 and p < 0.05 else ("HURTS" if ch > 0 and p < 0.05 else "no clear change")}  (change {ch:+.4f}, p {p:.4f})')

## 10. Figure and save

In [ ]:
labs = [l for k, l in ATTACKS if hv(f'{k}|std') and hv(f'{k}|def')]; ks = [k for k, l in ATTACKS if hv(f'{k}|std') and hv(f'{k}|def')]; xs = np.arange(len(ks))
fig, ax = plt.subplots(1, 2, figsize=(18, 5.2))
for off, mk, nm in [(-0.2, 'std', 'standard'), (0.2, 'def', 'defended')]:
    v = [A(f'{k}|{mk}', 'overall_evasion') for k in ks]; ax[0].bar(xs + off, [x.mean() for x in v], 0.4, yerr=[x.std(ddof=1) if len(x) > 1 else 0 for x in v], capsize=2, label=nm)
ax[0].set_xticks(xs); ax[0].set_xticklabels([l.replace(' (', '\n(') for l in labs], rotation=60, ha='right', fontsize=7); ax[0].set_ylabel('overall evasion of attack rows'); ax[0].set_title('Standard vs adversarially trained IDS'); ax[0].legend(); ax[0].grid(alpha=.3, axis='y')
wbk = [f'wb_{t}' for t in WB_BUDGETS if hv(f'wb_{t}|std')]
for c, cn in enumerate(atk_names):
    for mk, ls in [('std', '--'), ('def', '-')]: ax[1].plot([float(k.split('_')[1]) for k in wbk], [A(f'{k}|{mk}', 'per_class_evasion')[:, c].mean() for k in wbk], ls, marker='o', color=f'C{c}', label=f'{cn} ({mk})' if mk == 'def' else None)
ax[1].set_xlabel('white-box cyber budget (sigma)'); ax[1].set_ylabel('evasion'); ax[1].set_title('White-box (adaptive) per class: dashed = standard, solid = defended'); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'defence1_robustness_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'class_names': class_names, 'overall_table': overall_table, 'per_class_table': pc_table, 'per_seed': {s: dict(R[s]) for s in SEEDS}, 'info': INFO,
             'config': {'PGD_EPS': PGD_EPS, 'TRAIN_PPO_TAGS': TRAIN_PPO_TAGS, 'EVAL_PPO_TAGS': EVAL_PPO_TAGS, 'AUG_RATIO': AUG_RATIO, 'WB_BUDGETS': WB_BUDGETS, 'N_WB_PER_CLASS': N_WB_PER_CLASS, 'MAX_ITERS': MAX_ITERS}}, f'defence1_results_{PIPELINE_VERSION}.pkl')
overall_table.to_csv(f'defence1_overall_table_{PIPELINE_VERSION}.csv', index=False); pc_table.to_csv(f'defence1_per_class_table_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB5', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'sb3_contrib': sb3_contrib.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb5_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'defence1_results_{PIPELINE_VERSION}.pkl', f'defence1_overall_table_{PIPELINE_VERSION}.csv', f'defence1_per_class_table_{PIPELINE_VERSION}.csv', f'defence1_robustness_{PIPELINE_VERSION}.png', f'nb5_run_manifest_{PIPELINE_VERSION}.json'] + [f'xgb_defended_{PIPELINE_VERSION}_seed{s}.json' for s in SEEDS]:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)